In [1]:
import pandas as pd
import itertools
import random
import choix
from google import genai
from google.genai import types
import openai 
import os
from dotenv import load_dotenv
from typing import List, Optional, Dict, Union
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
import plotly.express as px
import plotly.graph_objects as go
import time
import anthropic

In [ ]:
class LLMClient:
    """
    Unified LLM client supporting multiple backends.
    
    Parameters
    ----------
    api_key : str, optional
        API key for the LLM service. If None, reads from environment
    model_name : str
        Model identifier (e.g., 'gemini-2.0-flash-exp', 'gpt-4o', 'claude-sonnet-4')
    provider : str, optional
        Force specific provider ('google', 'openai', 'anthropic'). 
        If None, infers from model_name
    """
    
    def __init__(
            self,
            api_key: Optional[str] = None,
            model_name: str = 'gemini-2.0-flash-exp',
            provider: Optional[str] = None):

        self.model_name = model_name
        self.provider = provider or self._infer_provider(model_name)
        self.api_key = api_key or self._get_api_key()
        self.client = self._initialize_client()
    
    def _infer_provider(self, model_name: str) -> str:
        """Infer provider from model name."""
        if 'gemini' in model_name.lower():
            return 'google'
        elif 'gpt' in model_name.lower():
            return 'openai'
        elif 'claude' in model_name.lower():
            return 'anthropic'
        else:
            raise ValueError(
                f"Cannot infer provider from model_name '{model_name}'. "
                "Please specify provider explicitly."
            )
    
    def _get_api_key(self) -> str:
        """Get API key from environment based on provider."""
        env_vars = {
            'google': 'GENAI_API_KEY',
            'openai': 'OPENAI_API_KEY',
            'anthropic': 'ANTHROPIC_API_KEY'
        }
        
        env_var = env_vars.get(self.provider)
        if not env_var:
            raise ValueError(f"Unknown provider: {self.provider}")
        
        api_key = os.getenv(env_var)
        if not api_key:
            raise ValueError(
                f"API key not found. Set {env_var} environment variable."
            )
        
        return api_key
    
    def _initialize_client(self):
        """Initialize the appropriate client."""
        if self.provider == 'google':
            from google import genai
            return genai.Client(api_key=self.api_key)
        
        elif self.provider == 'openai':
            from openai import OpenAI
            return OpenAI(api_key=self.api_key)
        
        elif self.provider == 'anthropic':
            from anthropic import Anthropic
            return Anthropic(api_key=self.api_key)
        else:
            raise ValueError(f"Unsupported provider: {self.provider}")
    
    def generate(
            self,
            prompt: str,
            system_message: Optional[str] = None,
            temperature: float = 0.0,
            max_tokens: int = 500) -> str:
        """
        Generate text using the LLM.
        
        Parameters
        ----------
        prompt : str
            User prompt
        system_message : str, optional
            System instruction
        temperature : float
            Sampling temperature
        max_tokens : int
            Maximum tokens to generate
            
        Returns
        -------
        str
            Generated text
        """
        if system_message is None:
            system_message = (
                "You are a precise and detail-oriented assistant specializing "
                "in analyzing text for specific concepts and constructs."
            )
        
        if self.provider == 'google':
            return self._generate_google(prompt, system_message, temperature)
        
        elif self.provider == 'openai':
            return self._generate_openai(prompt, system_message, temperature, max_tokens)
        
        elif self.provider == 'anthropic':
            return self._generate_anthropic(prompt, system_message, temperature, max_tokens)
    
    def _generate_google(
            self,
            prompt: str,
            system_message: str,
            temperature: float) -> str:
        """Generate using Google GenAI."""
        from google.genai import types
        
        response = self.client.models.generate_content(
            model=self.model_name,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=temperature
            ),
            contents=prompt
        )
        return response.text
    
    def _generate_openai(
            self,
            prompt: str,
            system_message: str,
            temperature: float,
            max_tokens: int) -> str:
        """Generate using OpenAI."""
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    
    def _generate_anthropic(
        self,
        prompt: str,
        system_message: str,
        temperature: float,
        max_tokens: int
    ) -> str:
        """Generate using Anthropic."""
        response = self.client.messages.create(
            model=self.model_name,
            system=system_message,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.content[0].text


def query_llm(
        prompt: str,
        model: str,
        system_message: Optional[str] = None,
        client: Optional[LLMClient] = None,
        **kwargs) -> str:
    """
    Convenience function to query an LLM.
    
    Parameters
    ----------
    prompt : str
        User prompt
    model : str
        Model name
    system_message : str, optional
        System instruction
    client : LLMClient, optional
        Existing client to reuse
    **kwargs
        Additional arguments for generation
        
    Returns
    -------
    str
        Generated text
    """
    if client is None:
        client = LLMClient(model_name=model)
    
    return client.generate(prompt, system_message, **kwargs)

In [ ]:
class Pairadigm:
    def __init__(self, 
                 data: pd.DataFrame, 
                 item_id_name: str, 
                 text_name: Optional[str] = None, 
                 cgcot_prompts: Optional[List[str]] = None, 
                 model_name: str = 'gemini-2.0-flash-exp', 
                 api_key: Optional[str] = None, 
                 target_concept: Optional[str] = None): 
        """
        Main class for Concept-Guided Chain-of-Thought (CGCoT) pairwise annotation.
        
        Supports flexible workflows:
        1. Start with raw items -> generate breakdowns -> pair -> annotate -> score -> validate
        2. Start with paired items -> generate breakdowns -> annotate -> score -> validate
        3. Start with human-annotated pairs -> generate breakdowns -> annotate -> score -> compare -> validate
        
        Parameters
        ----------
        data : pd.DataFrame
            Input data with items to compare
        item_id_name : str
            Column name for unique item identifiers
        text_name : str, optional
            Column name for item text/content
        cgcot_prompts : List[str], optional
            CGCoT prompt templates for breakdowns
        model_name : str, default='gemini-2.0-flash-exp'
            LLM model to use
        api_key : str, optional
            API key for LLM service
        target_concept : str, optional
            The concept to evaluate (e.g., "objectivity", "political bias")
        """
        
        # Validate inputs
        if not isinstance(data, pd.DataFrame):
            raise TypeError("data must be a pandas DataFrame")
        if item_id_name not in data.columns:
            raise ValueError(f"Column '{item_id_name}' not found in DataFrame")
        
        self.data = data.copy()
        self.item_id_name = item_id_name
        self.text_name = text_name
        self.cgcot_prompts = cgcot_prompts
        self.model_name = model_name
        self.target_concept = target_concept
        
        # Initialize LLM client
        self.client = LLMClient(api_key=api_key, model_name=model_name)
        
        # Initialize result storage
        self.pairwise_df: Optional[pd.DataFrame] = None
        self.scored_df: Optional[pd.DataFrame] = None
        self.validation_results: Optional[Dict] = None
    
    def generate_cgcot_breakdown(
            text: str,
            model: str,
            concept_prompts: List[str],
            client: LLMClient = None,
            rate_limit_per_minute=None) -> str:
        """
        Generate concept-specific breakdown for a given text using CGCoT prompts.
        
        The function iteratively applies each prompt, building context from
        previous responses to create a comprehensive analysis.
        
        Parameters
        ----------
        text : str
            The input text to analyze
        model : str
            Model name to use
        concept_prompts : List[str]
            Sequential CGCoT prompt templates
        client : LLMClient, optional
            Reusable client instance
        rate_limit_per_minute : int
            API rate limit
            
        Returns
        -------
        str
            Concatenated concept-specific breakdown
            
        Examples
        --------
        >>> prompts = ["Analyze: {text}", "Expand on: {previous_answers}"]
        >>> breakdown = generate_cgcot_breakdown("Sample text", "gpt-4o", prompts)
        """

        if client is None:
            client = LLMClient(model_name=model)
        
        breakdown = [f"Original Text: {text}"]
        prev_answers = []
        sleep_time = 0

        if rate_limit_per_minute:
            sleep_time = 60.0 / rate_limit_per_minute
        
        for i, prompt_template in enumerate(concept_prompts):
            # Format the prompt with text and previous answers
            full_prompt = prompt_template.format(
                text=text,
                previous_answers="\n".join(prev_answers)
            )
            
            # Query the LLM
            response = client.generate(full_prompt)
            prev_answers.append(response)
            breakdown.append(f"Prompt {i+1} response: {response}")
            
            # Rate limiting (except for last prompt)
            if i < len(concept_prompts) - 1:
                time.sleep(sleep_time)
        
        return "\n".join(breakdown)


    def generate_breakdowns_parallel(
            df: pd.DataFrame,
            cgcot_prompts: List[str],
            model: str = 'gemini-2.0-flash-exp',
            row_name: str = None,
            row_id: str = None,
            max_workers: int = 8) -> Dict[str, str]:
        """
        Generate CGCoT breakdowns for multiple items in parallel.
        
        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing items to analyze
        cgcot_prompts : List[str]
            CGCoT prompt templates
        model : str
            Model name to use
        row_name : str
            Column name for text content
        row_id : str
            Column name for unique identifiers
        max_workers : int
            Number of parallel workers
            
        Returns
        -------
        Dict[str, str]
            Mapping from row_id to breakdown
            
        Examples
        --------
        >>> df = pd.DataFrame({'id': [1, 2], 'text': ['Text A', 'Text B']})
        >>> prompts = load_cgcot_prompts('prompts.txt')
        >>> breakdowns = generate_breakdowns_parallel(df, prompts, row_name='text', row_id='id')
        """
        if row_name is None or row_id is None:
            raise ValueError("row_name and row_id must be specified")
        
        if row_name not in df.columns or row_id not in df.columns:
            raise ValueError(f"Columns '{row_name}' or '{row_id}' not found in DataFrame")
        
        # Create a shared client for efficiency
        client = LLMClient(model_name=model)
        
        results = {}
        total = len(df)
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks
            futures = {
                executor.submit(
                    generate_cgcot_breakdown,
                    row[row_name],
                    model,
                    cgcot_prompts,
                    client
                ): row[row_id]
                for _, row in df.iterrows()
            }
            
            # Collect results as they complete
            for i, future in enumerate(as_completed(futures), 1):
                item_id = futures[future]
                try:
                    results[item_id] = future.result()
                    if i % 10 == 0:
                        print(f"  Completed {i}/{total} breakdowns")
                except Exception as e:
                    print(f"  Error processing item {item_id}: {e}")
                    results[item_id] = f"ERROR: {e}"
        
        print(f"  Completed all {total} breakdowns")
        return results

In [ ]:
def load_cgcot_prompts(file_path: str) -> List[str]:
    """
    Load CGCoT prompt templates from a text file.
    
    Parameters
    ----------
    file_path : str
        Path to the text file containing prompts (one per line)
        
    Returns
    -------
    List[str]
        List of prompt templates
        
    Examples
    --------
    >>> prompts = load_cgcot_prompts('prompts.txt')
    >>> len(prompts)
    3
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        prompts = [line.strip() for line in file if line.strip()]
    
    if not prompts:
        raise ValueError(f"No prompts found in {file_path}")
    
    return prompts


def generate_cgcot_breakdown(
    text: str,
    model: str,
    concept_prompts: List[str],
    client: LLMClient = None,
    rate_limit_per_minute: int = 15
) -> str:
    """
    Generate concept-specific breakdown for a given text using CGCoT prompts.
    
    The function iteratively applies each prompt, building context from
    previous responses to create a comprehensive analysis.
    
    Parameters
    ----------
    text : str
        The input text to analyze
    model : str
        Model name to use
    concept_prompts : List[str]
        Sequential CGCoT prompt templates
    client : LLMClient, optional
        Reusable client instance
    rate_limit_per_minute : int
        API rate limit
        
    Returns
    -------
    str
        Concatenated concept-specific breakdown
        
    Examples
    --------
    >>> prompts = ["Analyze: {text}", "Expand on: {previous_answers}"]
    >>> breakdown = generate_cgcot_breakdown("Sample text", "gpt-4o", prompts)
    """
    if client is None:
        client = LLMClient(model_name=model)
    
    breakdown = [f"Original Text: {text}"]
    prev_answers = []
    sleep_time = 60.0 / rate_limit_per_minute
    
    for i, prompt_template in enumerate(concept_prompts):
        # Format the prompt with text and previous answers
        full_prompt = prompt_template.format(
            text=text,
            previous_answers="\n".join(prev_answers)
        )
        
        # Query the LLM
        response = client.generate(full_prompt)
        prev_answers.append(response)
        breakdown.append(f"Prompt {i+1} response: {response}")
        
        # Rate limiting (except for last prompt)
        if i < len(concept_prompts) - 1:
            time.sleep(sleep_time)
    
    return "\n".join(breakdown)


def generate_breakdowns_parallel(
    df: pd.DataFrame,
    cgcot_prompts: List[str],
    model: str = 'gemini-2.0-flash-exp',
    row_name: str = None,
    row_id: str = None,
    max_workers: int = 8
) -> Dict[str, str]:
    """
    Generate CGCoT breakdowns for multiple items in parallel.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing items to analyze
    cgcot_prompts : List[str]
        CGCoT prompt templates
    model : str
        Model name to use
    row_name : str
        Column name for text content
    row_id : str
        Column name for unique identifiers
    max_workers : int
        Number of parallel workers
        
    Returns
    -------
    Dict[str, str]
        Mapping from row_id to breakdown
        
    Examples
    --------
    >>> df = pd.DataFrame({'id': [1, 2], 'text': ['Text A', 'Text B']})
    >>> prompts = load_cgcot_prompts('prompts.txt')
    >>> breakdowns = generate_breakdowns_parallel(df, prompts, row_name='text', row_id='id')
    """
    if row_name is None or row_id is None:
        raise ValueError("row_name and row_id must be specified")
    
    if row_name not in df.columns or row_id not in df.columns:
        raise ValueError(f"Columns '{row_name}' or '{row_id}' not found in DataFrame")
    
    # Create a shared client for efficiency
    client = LLMClient(model_name=model)
    
    results = {}
    total = len(df)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {
            executor.submit(
                generate_cgcot_breakdown,
                row[row_name],
                model,
                cgcot_prompts,
                client
            ): row[row_id]
            for _, row in df.iterrows()
        }
        
        # Collect results as they complete
        for i, future in enumerate(as_completed(futures), 1):
            item_id = futures[future]
            try:
                results[item_id] = future.result()
                if i % 10 == 0:
                    print(f"  Completed {i}/{total} breakdowns")
            except Exception as e:
                print(f"  Error processing item {item_id}: {e}")
                results[item_id] = f"ERROR: {e}"
    
    print(f"  Completed all {total} breakdowns")
    return results


def validate_cgcot_prompts(prompts: List[str]) -> bool:
    """
    Validate that CGCoT prompts contain required placeholders.
    
    Parameters
    ----------
    prompts : List[str]
        List of prompt templates
        
    Returns
    -------
    bool
        True if prompts are valid
        
    Raises
    ------
    ValueError
        If prompts are invalid
    """
    if not prompts:
        raise ValueError("Prompts list is empty")
    
    # First prompt should reference {text}
    if '{text}' not in prompts[0]:
        raise ValueError("First prompt must contain {text} placeholder")
    
    # Subsequent prompts should reference {previous_answers}
    for i, prompt in enumerate(prompts[1:], 1):
        if '{previous_answers}' not in prompt:
            raise ValueError(
                f"Prompt {i+1} should contain {{previous_answers}} placeholder"
            )
    
    return True